# TTS Exp2 — Chunk Size Comparison (MOS Study)

**Mục tiêu:** Đo mức degradation chất lượng TTS khi cắt text thành chunk nhỏ trước khi synthesize, với TTFA đo từ lúc gửi query (bao gồm LLM streaming).

| | |
|---|---|
| Model | VieNeuTTS-0.3B GGUF INT4 (`pnnbao-ump/VieNeu-TTS-0.3B-q4-gguf`) |
| LLM | Qwen3-4B-Instruct (self-hosted vLLM, OpenAI-compatible API) |
| Conditions | 4: chunk_20 / chunk_40 / chunk_80 / full |
| Chunker | Copy chính xác từ `tts/core/chunker.py` — chỉ cắt tại `.!?` |
| Production min_size | 80 chars (từ `tts/grpc_handler.py: _CHUNK_MIN_CHARS=80`) |
| Queries | 250 câu hỏi dinh dưỡng stratified từ `synthetic_qa.jsonl` (seed=42) |
| Output | 1000 WAV files (250 queries × 4 conditions) |
| MOS design | Within-subject, 5 rater, 200 clips/người (1000 / 5) |

**TTFA đo đúng production:**
- `start` = lúc gửi query lên LLM
- LLM stream token → buffer → khi buffer đủ `min_size` chars và gặp `.!?` → đẩy chunk vào TTS ngay
- `ttfa_ms` = thời gian từ `start` đến khi TTS trả về frame audio đầu tiên
- `ttft_ms` = thời gian từ `start` đến LLM token đầu tiên (đo riêng để tách LLM vs TTS overhead)

**Pipeline:**
1. Load 250 queries từ `synthetic_qa.jsonl`
2. Với mỗi query × condition: LLM stream → buffer-chunk → TTS stream song song → WAV + latency
3. Ẩn danh + shuffle → `mos_clips/` + `mos_mapping.json`
4. (Sau khi có CSV) → Friedman test + biểu đồ MOS vs Latency

## 1. Install

- `llama-cpp-python` phải build với CUDA (bắt buộc cho GGUF backbone)
- `neucodec==0.0.3`, `phonemizer`, `librosa` là deps của vieneu (từ `tts/requirements.txt`)
- `openai` để gọi vLLM qua OpenAI-compatible API
- Không install vllm/lmdeploy trong notebook — vLLM chạy riêng trên server

In [ ]:
%%capture
# llama-cpp-python với CUDA — bắt buộc cho GGUF backbone
!CMAKE_ARGS='-DGGML_CUDA=on' pip install llama-cpp-python --upgrade --force-reinstall --no-cache-dir

# deps của vieneu (từ tts/requirements.txt)
!pip install 'neucodec==0.0.3' phonemizer librosa huggingface_hub transformers torch torchaudio soundfile

# OpenAI-compatible client để gọi vLLM
!pip install openai

# Analysis
!pip install pandas matplotlib scipy scikit-posthocs

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f'VRAM free: {free / 1e9:.1f} GB')

## 2. Mount Drive & paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_DIR  = '/content/drive/MyDrive/tts_experiment'
OUTPUT_DIR = Path(f'{DRIVE_DIR}/wav_exp2')
MOS_DIR    = Path(f'{DRIVE_DIR}/mos_clips')

for d in [OUTPUT_DIR, MOS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# vLLM endpoint — chạy trên server riêng, notebook chỉ gọi HTTP
# Nếu dùng ngrok/tunnel để expose: thay bằng URL tunnel
LLM_BASE_URL = os.environ.get('LLM_BASE_URL', 'http://localhost:8000/v1')
LLM_MODEL    = os.environ.get('LLM_MODEL',    'Qwen/Qwen3-4B-Instruct-2507')

print(f'Output  : {OUTPUT_DIR}')
print(f'MOS dir : {MOS_DIR}')
print(f'LLM URL : {LLM_BASE_URL}')
print(f'LLM model: {LLM_MODEL}')

## 3. Queries — 250 câu stratified từ `synthetic_qa.jsonl`

Stratified sampling theo source (seed=42), tỉ lệ giữ nguyên:
- **benhvienthucuc** × 78
- **viendinhduong** × 72
- **suckhoedoisong** × 51
- **vinmec** × 49

Tổng: 250 queries → 250 × 4 conditions = **1000 WAV files**.

In [ ]:
import json
import random

SYNTHETIC_QA_PATH = '/content/drive/MyDrive/tts_experiment/synthetic_qa.jsonl'

with open(SYNTHETIC_QA_PATH, encoding='utf-8') as f:
    records = [json.loads(line) for line in f if line.strip()]

by_source = {}
for r in records:
    by_source.setdefault(r['source'], []).append(r)

# Stratified sampling — tỉ lệ proportional, tổng = 250
SAMPLE_COUNTS = {
    'benhvienthucuc': 78,
    'viendinhduong':  72,
    'suckhoedoisong': 51,
    'vinmec':         49,
}
assert sum(SAMPLE_COUNTS.values()) == 250

random.seed(42)
selected = []
for src, n in SAMPLE_COUNTS.items():
    sampled = random.sample(by_source[src], n)
    selected.extend(sampled)
random.shuffle(selected)

TEST_QUERIES = [r['question'] for r in selected]
assert len(TEST_QUERIES) == 250

print(f'Total queries: {len(TEST_QUERIES)}')
from collections import Counter
src_counts = Counter(r['source'] for r in selected)
for src, n in sorted(src_counts.items(), key=lambda x: -x[1]):
    print(f'  {src}: {n}')
print()
for i, q in enumerate(TEST_QUERIES[:5], 1):
    print(f'Q{i:03d}: {q}')
print('...')

## 4. LLM Client — OpenAI-compatible (vLLM serving Qwen3-4B)

Copy interface từ `brain/core/llm.py: OpenAILLMClient`.  
`generate_stream()` yield từng token — dùng để buffer-chunk-TTS song song.

In [ ]:
import re
import time
from openai import AsyncOpenAI

# Copy chính xác từ brain/core/prompt.py: NUTRITION_SYSTEM_PROMPT
SYSTEM_PROMPT = """Bạn là chuyên gia tư vấn dinh dưỡng qua giọng nói. Tuân thủ:

1. **Dựa vào tài liệu**: Trả lời dựa trên thông tin dinh dưỡng được cung cấp. Không trích dẫn tên nguồn hay URL trong câu trả lời.
2. **Phong cách bác sĩ**: Bắt đầu bằng "Chào bạn,", tư vấn như chuyên gia dinh dưỡng.
3. **Ngắn gọn, dễ nghe**: Câu trả lời sẽ được đọc thành giọng nói — tối đa 150 từ, dùng câu ngắn, không dùng bullet points hay danh sách. Sau mỗi dấu chấm hoặc dấu phẩy phải có dấu cách.
4. **Trung thực**: Nếu không có thông tin → nói rõ "Tôi không có thông tin về vấn đề này".
5. **Disclaimer**: Kết thúc bằng "Để được tư vấn chính xác, bạn nên gặp bác sĩ dinh dưỡng."
"""

llm_client = AsyncOpenAI(base_url=LLM_BASE_URL, api_key='dummy')


async def llm_stream(query: str):
    """
    Yield từng text token từ vLLM.
    Copy từ brain/core/llm.py: OpenAILLMClient.generate_stream()
    enable_thinking=False để tắt chain-of-thought của Qwen3.
    """
    stream = await llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': query},
        ],
        temperature=0.3,
        max_tokens=400,
        stream=True,
        extra_body={'chat_template_kwargs': {'enable_thinking': False}},
    )
    async for chunk in stream:
        text = chunk.choices[0].delta.content
        if text:
            yield text


# Ping test — verify LLM endpoint trả về response
print('Testing LLM endpoint...')
test_tokens = []
async for tok in llm_stream('Canxi có trong thực phẩm nào?'):
    test_tokens.append(tok)
    if len(test_tokens) == 1:
        print('  First token received ✓')
    if len(''.join(test_tokens)) > 80:
        break
print(f'  Sample: {"".join(test_tokens)[:100]}...')
print('LLM OK.')

## 5. Load TTS model

Dùng `Vieneu` (không phải `VieNeuTTS`) và params đúng với `tts/core/synthesizer.py`:
- `backbone_device='gpu'` → llama-cpp với CUDA
- `codec_device='cuda'` → neucodec với CUDA
- `temperature=1.0, top_k=50` trong `infer_stream()`

In [ ]:
from vieneu import Vieneu

print('Loading VieNeuTTS-0.3B GGUF INT4...')
tts = Vieneu(
    backbone_repo='pnnbao-ump/VieNeu-TTS-0.3B-q4-gguf',
    backbone_device='gpu',
    codec_repo='neuphonic/distill-neucodec',
    codec_device='cuda',
)
voice = tts.get_preset_voice()

# Warmup — tránh đo cold start latency
print('Warming up...')
_ = list(tts.infer_stream('Xin chào.', voice=voice, temperature=1.0, top_k=50))
print('TTS ready.')

## 6. Chunker — copy chính xác từ `tts/core/chunker.py`

**Quan trọng:** Production chunker chỉ cắt tại `.!?` — không cắt tại `,;:`.
Comment trong code: *'TTS model thêm prosody kết thúc sau mỗi chunk — nếu chunk kết thúc tại dấu phẩy thì nghe như ngắt câu tùy tiện.'*

| Condition | min_size | Ghi chú |
|---|---|---|
| chunk_20 | 20 | TTFA tốt nhất, quality thấp nhất |
| chunk_40 | 40 | Thử nghiệm |
| chunk_80 | 80 | **Production default** (`grpc_handler.py: _CHUNK_MIN_CHARS=80`) |
| full | — | Không cắt, baseline chất lượng tốt nhất |

In [ ]:
import numpy as np
import soundfile as sf
import threading
import asyncio
import statistics
from typing import List, Optional

# ── Chunker — copy chính xác từ tts/core/chunker.py ──────────────────────────
_SENT_END   = re.compile(r'[.!?]')
_NEWLINE_RE = re.compile(r'\n+')
SAMPLE_RATE = 24_000


def chunk_text(text: str, min_size: int) -> List[str]:
    """Từ tts/core/chunker.py — chỉ cắt tại .!?, không cắt tại ,;:"""
    chunks, buffer = [], ''
    for char in text:
        buffer += char
        if _SENT_END.match(char) and len(buffer.strip()) >= min_size:
            chunks.append(buffer)
            buffer = ''
    if buffer.strip():
        chunks.append(buffer)
    return chunks


def chunk_full(text: str) -> List[str]:
    return [text.strip()]


def pcm_to_int16_bytes(audio: np.ndarray) -> bytes:
    """Từ tts/core/synthesizer.py: synthesize_stream() line 126."""
    return (audio * 32767).clip(-32768, 32767).astype(np.int16).tobytes()


def save_wav(pcm_chunks: list, path) -> float:
    raw   = b''.join(pcm_chunks)
    audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    sf.write(str(path), audio, SAMPLE_RATE)
    return len(audio) / SAMPLE_RATE


# ── TTS helper — thread + asyncio.Queue bridge (từ tts/grpc_handler.py) ───────

async def _tts_chunk(text: str, tts_instance, voice, loop) -> List[bytes]:
    """
    Synthesize 1 chunk, trả về list PCM frames.
    Params từ tts/core/synthesizer.py: Synthesizer.synthesize_stream()
      - temperature=1.0, top_k=50, max_chars=256
    """
    q: asyncio.Queue = asyncio.Queue()

    def _worker():
        try:
            for frame in tts_instance.infer_stream(
                text=text,
                voice=voice,
                max_chars=256,
                temperature=1.0,
                top_k=50,
            ):
                loop.call_soon_threadsafe(q.put_nowait, frame)
        except Exception as e:
            loop.call_soon_threadsafe(q.put_nowait, e)
        finally:
            loop.call_soon_threadsafe(q.put_nowait, None)

    threading.Thread(target=_worker, daemon=True).start()
    frames = []
    while True:
        frame = await q.get()
        if frame is None:
            break
        if isinstance(frame, Exception):
            print(f'  TTS error: {frame}')
            break
        frames.append(pcm_to_int16_bytes(frame))
    return frames


# ── Core pipeline: LLM stream → buffer-chunk → TTS ───────────────────────────

async def run_llm_stream_tts(
    query: str,
    min_size: Optional[int],  # None = full (không chunk, chờ LLM xong)
    tts_instance,
    voice,
) -> dict:
    """
    Stream LLM tokens, buffer đến khi đủ min_size chars + gặp .!? → TTS ngay.
    Nếu min_size is None: thu thập full response rồi mới TTS (condition 'full').

    TTFT = lúc gửi query → LLM token đầu tiên (LLM overhead)
    TTFA = lúc gửi query → PCM frame TTS đầu tiên (LLM + TTS chunk đầu)
    Total = lúc gửi query → PCM frame TTS cuối cùng
    """
    loop    = asyncio.get_running_loop()
    pcm_all = []
    ttft_ms: Optional[float] = None
    ttfa_ms: Optional[float] = None
    n_chunks = 0
    full_text_parts = []
    start = time.perf_counter()

    if min_size is None:
        # ── condition 'full': chờ LLM hoàn toàn rồi mới TTS ─────────────────
        async for token in llm_stream(query):
            if ttft_ms is None:
                ttft_ms = (time.perf_counter() - start) * 1000
            full_text_parts.append(token)

        full_text = _NEWLINE_RE.sub(' ', ''.join(full_text_parts)).strip()
        if full_text:
            frames = await _tts_chunk(full_text, tts_instance, voice, loop)
            if frames and ttfa_ms is None:
                ttfa_ms = (time.perf_counter() - start) * 1000
            pcm_all.extend(frames)
            n_chunks = 1

    else:
        # ── condition chunk_N: buffer token, flush khi đủ min_size + .!? ──────
        buffer = ''
        async for token in llm_stream(query):
            if ttft_ms is None:
                ttft_ms = (time.perf_counter() - start) * 1000
            full_text_parts.append(token)
            buffer += token

            if _SENT_END.search(buffer) and len(buffer.strip()) >= min_size:
                chunk = buffer.strip()
                buffer = ''
                frames = await _tts_chunk(chunk, tts_instance, voice, loop)
                if frames and ttfa_ms is None:
                    ttfa_ms = (time.perf_counter() - start) * 1000
                pcm_all.extend(frames)
                n_chunks += 1

        # Flush phần còn lại sau khi LLM kết thúc
        if buffer.strip():
            frames = await _tts_chunk(buffer.strip(), tts_instance, voice, loop)
            if frames and ttfa_ms is None:
                ttfa_ms = (time.perf_counter() - start) * 1000
            pcm_all.extend(frames)
            n_chunks += 1

        full_text = _NEWLINE_RE.sub(' ', ''.join(full_text_parts)).strip()

    total_ms = (time.perf_counter() - start) * 1000
    return {
        'pcm_all':   pcm_all,
        'full_text': full_text,
        'n_chunks':  n_chunks,
        'ttft_ms':   round(ttft_ms, 1) if ttft_ms else None,
        'ttfa_ms':   round(ttfa_ms, 1) if ttfa_ms else None,
        'total_ms':  round(total_ms, 1),
    }


# ── Verify chunker ────────────────────────────────────────────────────────────
sample = 'Chào bạn, canxi có nhiều trong sữa. Ngoài ra còn có rau xanh. Hải sản cũng là nguồn tốt!'
for ms in [20, 40, 80]:
    cs = chunk_text(sample, ms)
    print(f'min={ms}: {len(cs)} chunks → {cs}')
print(f'full: {chunk_full(sample)}')

## 7. Run — LLM stream → buffer-chunk → TTS (4 conditions × 50 queries)

**TTFA** = từ lúc gửi query đến frame audio đầu tiên (bao gồm LLM latency + TTS latency của chunk đầu).  
**TTFT** = từ lúc gửi query đến token LLM đầu tiên (để tách overhead LLM vs TTS).  
**Total** = từ lúc gửi query đến frame audio cuối cùng.  

Với `chunk_20`: TTFA ≈ TTFT + TTS(20 chars) — nhanh nhất.  
Với `full`: TTFA ≈ LLM_total + TTS(full_text) — chậm nhất.

In [ ]:
# min_size=None → condition 'full' (chờ LLM xong mới TTS)
CONDITIONS = [
    ('chunk_20', 20),
    ('chunk_40', 40),
    ('chunk_80', 80),    # production
    ('full',     None),  # baseline
]

all_results = []

for cond_name, min_size in CONDITIONS:
    print(f'\n--- {cond_name} ---')
    cond_dir = OUTPUT_DIR / cond_name
    cond_dir.mkdir(exist_ok=True)

    ttfts, ttfas, totals = [], [], []

    for q_idx, query in enumerate(TEST_QUERIES):
        result = await run_llm_stream_tts(query, min_size, tts, voice)

        wav_path   = cond_dir / f'{cond_name}_q{q_idx+1:02d}.wav'
        duration_s = save_wav(result['pcm_all'], wav_path) if result['pcm_all'] else 0.0

        row = {
            'condition':      cond_name,
            'query_idx':      q_idx + 1,
            'query':          query,
            'n_chunks':       result['n_chunks'],
            'response_words': len(result['full_text'].split()),
            'ttft_ms':        result['ttft_ms'],
            'ttfa_ms':        result['ttfa_ms'],
            'total_ms':       result['total_ms'],
            'duration_s':     round(duration_s, 2),
            'wav_path':       str(wav_path),
        }
        all_results.append(row)

        if result['ttft_ms']: ttfts.append(result['ttft_ms'])
        if result['ttfa_ms']: ttfas.append(result['ttfa_ms'])
        totals.append(result['total_ms'])

        print(
            f"  Q{q_idx+1:02d} [{result['n_chunks']} chunk]"
            f"  TTFT={result['ttft_ms']:.0f}ms"
            f"  TTFA={result['ttfa_ms']:.0f}ms"
            f"  Total={result['total_ms']:.0f}ms"
            f"  Dur={duration_s:.1f}s"
        )

    p95 = lambda lst: sorted(lst)[min(int(len(lst) * 0.95), len(lst) - 1)] if lst else 0
    print(f'  TTFT  median={statistics.median(ttfts):.0f}ms  p95={p95(ttfts):.0f}ms')
    if ttfas:
        print(f'  TTFA  median={statistics.median(ttfas):.0f}ms  p95={p95(ttfas):.0f}ms')
    print(f'  Total median={statistics.median(totals):.0f}ms  p95={p95(totals):.0f}ms')

lat_path = f'{DRIVE_DIR}/exp2_latency_results.json'
with open(lat_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f'\nLatency saved: {lat_path}')
print(f'WAV files: {len(list(OUTPUT_DIR.rglob("*.wav")))}')

## 8. Latency summary

In [ ]:
import pandas as pd

df  = pd.DataFrame(all_results)
ord = ['chunk_20', 'chunk_40', 'chunk_80', 'full']

summary = df.groupby('condition').agg(
    n_chunks_avg =('n_chunks',  'mean'),
    ttft_median  =('ttft_ms',   'median'),
    ttft_p95     =('ttft_ms',   lambda x: x.quantile(0.95)),
    ttfa_median  =('ttfa_ms',   'median'),
    ttfa_p95     =('ttfa_ms',   lambda x: x.quantile(0.95)),
    total_median =('total_ms',  'median'),
    total_p95    =('total_ms',  lambda x: x.quantile(0.95)),
).round(1).reindex(ord)

print('Latency Summary (ms):')
print(summary.to_string())
print()
print('TTFT ≈ LLM latency (giống nhau cho 4 conditions — kiểm tra nếu chênh lệch lớn)')
print('TTFA - TTFT ≈ TTS latency của chunk đầu tiên')

## 9. Verify — nghe Q8 (response dài) cả 4 conditions

Q8 = *'Thế nào là một bữa ăn đủ chất dinh dưỡng và cân đối?'* — response dài nhất, thể hiện rõ nhất sự khác biệt giữa chunk sizes.

In [ ]:
from IPython.display import Audio, display

# Chọn query có response dài nhất để thể hiện rõ sự khác biệt chunk size
try:
    longest = max(
        (r for r in all_results if r['condition'] == 'full'),
        key=lambda r: r['response_words']
    )
    Q_VERIFY = longest['query_idx']
except Exception:
    Q_VERIFY = 8

print(f'Q{Q_VERIFY}: {TEST_QUERIES[Q_VERIFY - 1]}')
print()

for cond_name, min_size in CONDITIONS:
    p = OUTPUT_DIR / cond_name / f'{cond_name}_q{Q_VERIFY:02d}.wav'
    row = next((r for r in all_results
                if r['condition'] == cond_name and r['query_idx'] == Q_VERIFY), None)
    if p.exists() and row:
        print(f'[{cond_name}] {row["n_chunks"]} chunk(s) | '
              f'TTFT={row["ttft_ms"]}ms  TTFA={row["ttfa_ms"]}ms  Total={row["total_ms"]}ms')
        print(f'  Response: {row["response_words"]} từ')
        audio, sr = sf.read(str(p))
        display(Audio(audio, rate=sr))
    else:
        print(f'[{cond_name}] WAV not found: {p}')

## 10. Ẩn danh + shuffle clips cho MOS

Rater không biết condition → tránh bias.
Mapping `clip_id → (condition, query_idx)` lưu riêng để giải mã sau.

In [ ]:
import random
import shutil
from collections import Counter

random.seed(42)

clips = [
    r for r in all_results
    if Path(r['wav_path']).exists() and Path(r['wav_path']).stat().st_size > 2000
]
print(f'Valid clips: {len(clips)} / {len(all_results)}  (expected 1000 = 250 queries × 4 conditions)')
if len(clips) < len(all_results):
    missing = [r for r in all_results if r not in clips]
    print('Missing:', [m['wav_path'] for m in missing])

random.shuffle(clips)

mapping = {}
for i, r in enumerate(clips, 1):
    anon = f'clip_{i:04d}.wav'
    shutil.copy2(r['wav_path'], MOS_DIR / anon)
    mapping[anon] = {
        'condition':      r['condition'],
        'query_idx':      r['query_idx'],
        'query':          r['query'],
        'response_words': r['response_words'],
        'n_chunks':       r['n_chunks'],
    }

mapping_path = f'{DRIVE_DIR}/mos_mapping.json'
with open(mapping_path, 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print(f'MOS clips: {MOS_DIR}')
print(f'Mapping  : {mapping_path}')
counts = Counter(v['condition'] for v in mapping.values())
for c in ['chunk_20', 'chunk_40', 'chunk_80', 'full']:
    print(f'  {c}: {counts.get(c, 0)} clips')

## 11. Hướng dẫn rater & setup Google Forms

Copy phần instructions gửi cho rater. Setup Forms:
1. Upload `mos_clips/` lên Drive, set *'Anyone with link can view'*
2. Tạo 1 Google Form, thêm 3 practice clips đầu (không tính điểm)
3. Mỗi clip = 1 câu hỏi: tiêu đề là tên file, mô tả là link Drive, scale 1–5
4. Bắt buộc trả lời từng câu

In [ ]:
instructions = '''
=== HƯỚNG DẪN ĐÁNH GIÁ CHẤT LƯỢNG GIỌNG ĐỌC ===

Bạn sẽ nghe 200 đoạn audio tiếng Việt (câu trả lời tư vấn dinh dưỡng do AI tổng hợp).
Với mỗi đoạn, chấm điểm 3 tiêu chí từ 1 đến 5:

  NATURALNESS (Tự nhiên):
    5 — Hoàn toàn tự nhiên, như người thật nói
    4 — Tự nhiên, đôi chỗ nhỏ nghe hơi máy
    3 — Chấp nhận được, nghe rõ nội dung nhưng rõ là giọng máy
    2 — Khó nghe, không tự nhiên nhiều chỗ
    1 — Rất khó nghe

  PROSODY (Ngữ điệu):
    5 — Nhấn đúng trọng âm, lên xuống giọng rất tự nhiên
    4 — Ngữ điệu tốt, đôi chỗ hơi cứng
    3 — Ngữ điệu trung bình, nghe được
    2 — Nhấn sai trọng âm hoặc đọc đều đều, thiếu cảm xúc
    1 — Ngữ điệu rất sai hoặc gần như không có

  CONTINUITY (Liền mạch):
    5 — Toàn bộ đoạn audio liền mạch, không ngắt quãng
    4 — Liền mạch, có 1–2 chỗ dừng nhỏ không đáng kể
    3 — Có vài chỗ ngắt nhưng vẫn theo dõi được
    2 — Ngắt nhiều chỗ, khó theo dõi
    1 — Giật cục, gần như không thể nghe liên tục

Lưu ý:
- Nghe lại tối đa 2 lần trước khi chấm
- 3 clip đầu là practice để calibrate — không tính vào điểm
- Yêu cầu: tai nghe, phòng yên tĩnh, tiếng Việt bản ngữ
- Thời gian ước tính: ~90–120 phút
'''
print(instructions)

print('=== CLIP LIST ===')
for name in sorted(mapping.keys()):
    print(f'  {name}')

---
## 12. Phân tích MOS — chạy sau khi có CSV từ Google Forms

Google Forms export CSV:
- Mỗi hàng = 1 rater
- Cột: `Timestamp`, rồi mỗi câu hỏi clip = 1 cột
- Giá trị: điểm 1–5

Download CSV → upload vào `DRIVE_DIR/mos_responses.csv` → chạy cells dưới.

In [ ]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

# Re-define paths (in case running analysis in a fresh session)
DRIVE_DIR    = '/content/drive/MyDrive/tts_experiment'
CSV_PATH     = f'{DRIVE_DIR}/mos_responses.csv'
mapping_path = f'{DRIVE_DIR}/mos_mapping.json'
lat_path     = f'{DRIVE_DIR}/exp2_latency_results.json'

df_raw = pd.read_csv(CSV_PATH)
print(f'Shape: {df_raw.shape}  ({len(df_raw)} rater)')
print(df_raw.head(2))

In [ ]:
# Parse: melt thành long format
clip_cols = [c for c in df_raw.columns if 'clip_' in c]

# Google Forms đôi khi thêm text dài vào tên cột — extract clip_XXX
rename = {}
for col in clip_cols:
    m = re.search(r'clip_\d{3}', col)
    if m:
        rename[col] = m.group()
df_scores = df_raw[clip_cols].rename(columns=rename)

df_long = df_scores.melt(var_name='clip', value_name='score')
df_long['score'] = pd.to_numeric(df_long['score'], errors='coerce')
df_long = df_long.dropna(subset=['score'])

# Join với mapping
with open(mapping_path, encoding='utf-8') as f:
    mos_map = json.load(f)

df_long['condition'] = df_long['clip'].map(
    lambda c: mos_map.get(c + '.wav', {}).get('condition')
)
df_long['query_idx'] = df_long['clip'].map(
    lambda c: mos_map.get(c + '.wav', {}).get('query_idx')
)

print(f'Total scores: {len(df_long)}')
print(df_long.groupby('condition')['score'].agg(['count', 'mean']).round(2))

In [ ]:
order = ['chunk_20', 'chunk_40', 'chunk_80', 'full']

# Mean MOS ± 95% CI
mos_stats = []
print('MOS Results:')
for cond in order:
    scores = df_long[df_long['condition'] == cond]['score'].dropna()
    n, mean = len(scores), scores.mean()
    ci95 = (scores.std() / np.sqrt(n)) * stats.t.ppf(0.975, df=n - 1)
    mos_stats.append({'condition': cond, 'mean': mean, 'ci95': ci95, 'n': n})
    print(f'  {cond:10s}: {mean:.2f} ± {ci95:.2f}  (n={n})')

# Friedman test — within-subject non-parametric
# pivot: mỗi hàng = 1 clip, mỗi cột = 1 condition (aggregate qua rater)
pivot = df_long.pivot_table(index='clip', columns='condition', values='score', aggfunc='mean')
pivot = pivot[order].dropna()
stat, p_val = stats.friedmanchisquare(*[pivot[c].values for c in order])
print(f'\nFriedman: chi2={stat:.3f}, p={p_val:.4f}')
print('→ Significant (p<0.05)' if p_val < 0.05 else '→ Not significant (p>=0.05)')

In [ ]:
try:
    from scikit_posthocs import posthoc_dunn
    df_ph = df_long[df_long['condition'].isin(order)][['condition', 'score']].dropna()
    dunn  = posthoc_dunn(df_ph, val_col='score', group_col='condition', p_adjust='holm')
    print('Dunn post-hoc (Holm-corrected p-values):')
    print(dunn.reindex(order)[order].round(4))
except ImportError:
    print('Run: pip install scikit-posthocs')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

labels = ['20 chars', '40 chars', '80 chars\n(production)', 'Full\n(baseline)']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
order  = ['chunk_20', 'chunk_40', 'chunk_80', 'full']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ax1, ax2, ax3 = axes

# ── MOS bar ───────────────────────────────────────────────────────────────────
means = [s['mean'] for s in mos_stats]
cis   = [s['ci95'] for s in mos_stats]
bars  = ax1.bar(labels, means, yerr=cis, capsize=6,
                color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)
ax1.set_ylim(1, 5.5)
ax1.set_ylabel('MOS (1–5)', fontsize=12)
ax1.set_title('Mean MOS ± 95% CI by Chunk Size', fontsize=13)
for bar, m in zip(bars, means):
    ax1.text(bar.get_x() + bar.get_width()/2, m + 0.12,
             f'{m:.2f}', ha='center', fontsize=10, fontweight='bold')
ax1.axhline(means[2], color=colors[2], linestyle='--', alpha=0.4, label='Production (80)')
ax1.legend(fontsize=9)

# ── Load latency ──────────────────────────────────────────────────────────────
try:
    _lat = all_results
except NameError:
    with open(lat_path, encoding='utf-8') as f:
        _lat = json.load(f)

df_lat     = pd.DataFrame(_lat)
ttft_med   = df_lat.groupby('condition')['ttft_ms'].median().reindex(order)
ttfa_med   = df_lat.groupby('condition')['ttfa_ms'].median().reindex(order)
total_med  = df_lat.groupby('condition')['total_ms'].median().reindex(order)
x = np.arange(len(order))
w = 0.25

# ── TTFA breakdown bar (TTFT + TTS overhead) ──────────────────────────────────
tts_overhead = ttfa_med.values - ttft_med.values  # TTFA - TTFT ≈ TTS latency chunk đầu
ax2.bar(x, ttft_med.values,  w * 3, label='TTFT (LLM)',        color='#9B59B6', alpha=0.85)
ax2.bar(x, tts_overhead,     w * 3, label='TTS overhead',      color='#E74C3C', alpha=0.75,
        bottom=ttft_med.values)
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.set_ylabel('Latency (ms)', fontsize=12)
ax2.set_title('TTFA Breakdown\n(TTFT + TTS chunk đầu)', fontsize=13)
ax2.legend(fontsize=9)
for i, (tf, ttfa) in enumerate(zip(ttft_med.values, ttfa_med.values)):
    ax2.text(i, ttfa + 20, f'{ttfa:.0f}ms', ha='center', fontsize=9, fontweight='bold')

# ── Total latency ─────────────────────────────────────────────────────────────
ax3.bar(x - w/2, ttfa_med.values,  w, label='TTFA (ms)',  color='#4C72B0', alpha=0.85)
ax3.bar(x + w/2, total_med.values, w, label='Total (ms)', color='#55A868', alpha=0.85)
ax3.set_xticks(x)
ax3.set_xticklabels(labels)
ax3.set_ylabel('Latency (ms)', fontsize=12)
ax3.set_title('TTFA vs Total Latency\n(median)', fontsize=13)
ax3.legend(fontsize=9)

plt.suptitle('TTS Exp2: Chunk Size — Quality vs Latency Trade-off', fontsize=14, y=1.02)
plt.tight_layout()
fig_path = f'{DRIVE_DIR}/exp2_results.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')